# 05_bronze_validation

## Purpose

Validate the Day 2 Bronze layer after ingestion.

This notebook checks that all expected Bronze tables exist, contain rows, expose schemas, and include ingestion metadata for traceability.

In [0]:
from pyspark.sql import Row
from pyspark.sql import functions as F

catalog = "vattenfall_dev"
schema = "raw"

bronze_tables = [
    f"{catalog}.{schema}.bronze_market_prices",
    f"{catalog}.{schema}.bronze_weather",
    f"{catalog}.{schema}.bronze_grid_events",
    f"{catalog}.{schema}.bronze_asset_reference",
]

required_metadata_columns = ["ingestion_ts", "source_file"]

In [0]:
validation_rows = []

for table_name in bronze_tables:
    df = spark.table(table_name)
    columns = df.columns

    missing_metadata_columns = [
        column_name
        for column_name in required_metadata_columns
        if column_name not in columns
    ]

    validation_rows.append(
        Row(
            table_name=table_name,
            row_count=df.count(),
            column_count=len(columns),
            has_ingestion_ts="ingestion_ts" in columns,
            has_source_file="source_file" in columns,
            missing_metadata_columns=", ".join(missing_metadata_columns)
        )
    )

validation_summary_df = spark.createDataFrame(validation_rows)

display(validation_summary_df)

In [0]:
failed_tables = (
    validation_summary_df
    .filter(
        (F.col("row_count") <= 0)
        | (F.col("has_ingestion_ts") == False)
        | (F.col("has_source_file") == False)
    )
)

if failed_tables.count() > 0:
    display(failed_tables)
    raise ValueError("Bronze validation failed. One or more tables are empty or missing required metadata columns.")

print("Bronze validation passed for all expected tables.")

In [0]:
for table_name in bronze_tables:
    print(f"\nSchema for {table_name}")
    spark.table(table_name).printSchema()

In [0]:
for table_name in bronze_tables:
    print(f"\nSample rows for {table_name}")
    display(spark.table(table_name).limit(20))

In [0]:
for table_name in bronze_tables:
    print(f"\nSource file distribution for {table_name}")

    (
        spark.table(table_name)
        .groupBy("source_file")
        .count()
        .orderBy(F.desc("count"))
        .show(truncate=False)
    )